# 01 — Set up Jouvence and navigate its data model

This first lesson starts from a clean clone. It explains local setup, bounded fixture and live access, cloud identity and requester-pays, then follows one biological question across nodes, assertions, and source evidence. The course is read-only: no cell writes canonical data, and a bounded sample does not prove absence or completeness.


In [ ]:
from __future__ import annotations
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from manage_db.public_notebooks import (
    PUBLIC_KG_ROOT,
    build_public_fixture,
    parquet_catalog,
    read_bounded_parquet,
)

MODE = os.environ.get("JOUVENCE_DATA_MODE", "fixture")
BILLING_PROJECT = os.environ.get("JOUVENCE_BILLING_PROJECT")
CACHE = Path(os.environ.get("JOUVENCE_NOTEBOOK_CACHE", REPO_ROOT / "artifacts" / "cache" / "public-notebooks"))
CACHE.mkdir(parents=True, exist_ok=True)
KG_ROOT = build_public_fixture(CACHE / "kg-fixture") if MODE == "fixture" else PUBLIC_KG_ROOT
print({"mode": MODE, "kg_root": str(KG_ROOT), "bounded": True})


## Install the project and select its kernel

Jouvence uses Python 3.11 or newer and `uv` for a reproducible environment. From the repository root, install notebook dependencies and register or select the Python kernel created for this project. Running from the root also makes relative documentation and fixture paths predictable.


In [ ]:
setup_commands = [
    "uv sync --group dev --group notebooks --group gnn",
    "uv run python -m ipykernel install --user --name jouvence --display-name 'Jouvence (uv)'",
    "uv run python scripts/verify_data_quickstart.py --mode fixture",
]
print("Run these in a terminal from REPO_ROOT:")
print("\n".join(f"  {command}" for command in setup_commands))


### Interpretation

The notebook kernel and the shell environment must refer to the same checkout. A successful fixture quickstart validates imports and Parquet mechanics without requiring a cloud account, credentials, network access, or a billing project.


In [ ]:
runtime = {
    "python": sys.version.split()[0],
    "repository": str(REPO_ROOT),
    "mode": MODE,
    "fixture_exists": Path(KG_ROOT).exists() if MODE == "fixture" else None,
}
print(runtime)


### Checkpoint

Confirm that the repository path is this checkout and that fixture mode is selected before continuing. Notebook 02 reuses the same environment for embeddings.


## Choose fixture or bounded live mode

Fixture mode is deterministic and synthetic; it is the default for learning and CI. Live mode reads named objects from canonical Parquet only after the caller opts in with `JOUVENCE_DATA_MODE=live`. Live access is bounded and read-only, never a full-KG scan from a laptop.


In [ ]:
mode_contract = {
    "fixture": "local synthetic examples; no cloud access",
    "live": "named canonical GCS objects; caller credentials and billing required",
}
assert MODE in mode_contract, f"Unsupported JOUVENCE_DATA_MODE={MODE!r}"
print(MODE, "—", mode_contract[MODE])


### Interpretation

Fixture rows illustrate schemas and joins but are not release cardinalities or biological findings. A live bounded prefix is real data, yet it is still not a completeness query: a missing row may lie outside the selected row groups.


In [ ]:
if MODE == "live" and not BILLING_PROJECT:
    raise RuntimeError("Live mode requires the caller-owned JOUVENCE_BILLING_PROJECT")
print({"live_opt_in": MODE == "live", "row_ceiling": 10_000, "read_only": True})


### Checkpoint

Use fixture mode first. Switch to live mode only for a named table and a clear question, then keep limits small and selected columns narrow.


## Understand ADC, IAM, requester-pays, and quota

Application Default Credentials (ADC) are the local identity discovered by Google client libraries. Authentication proves who the caller is; IAM authorization decides whether that identity may read an object; requester-pays attributes request and egress charges to a caller-owned billing project; quota attribution determines which project consumes API quota. These are four separate gates.


In [ ]:
cloud_layers = [
    ("authentication", "gcloud auth application-default login", "establish caller identity"),
    ("authorization", "bucket IAM objectViewer grant", "allow object get/list"),
    ("billing", "billing-enabled caller project", "pay requester-pays charges"),
    ("quota", "serviceusage.services.use", "consume enabled API quota"),
]
import pandas as pd
display(pd.DataFrame(cloud_layers, columns=["gate", "example", "purpose"]))


### Interpretation

A valid ADC token does not imply bucket authorization, and bucket authorization does not imply permission to charge a billing project. Error messages should be mapped to the failed gate rather than worked around with embedded credentials.


In [ ]:
safe_cloud_commands = [
    "gcloud projects create <caller-owned-project>",
    "gcloud billing projects link <caller-owned-project> --billing-account=<caller-billing-account>",
    "gcloud services enable storage.googleapis.com --project=<caller-owned-project>",
    "gcloud auth application-default login",
    "export JOUVENCE_BILLING_PROJECT=<caller-owned-project>",
]
print("Illustrative commands; substitute only your own authorized project:\n" + "\n".join(safe_cloud_commands))


### Checkpoint

Never paste a credential, access token, service-account JSON, maintainer project ID, or billing account into a notebook. The maintainer must separately grant your identity read-only bucket access.


## Troubleshoot live access without increasing scope

Fail closed and diagnose layer by layer. `401` usually indicates authentication; `403` may indicate bucket IAM or `serviceusage.services.use`; requester-pays errors indicate missing billing attribution; API-disabled errors require enabling the Cloud Storage JSON API in the caller project.


In [ ]:
troubleshooting = pd.DataFrame([
    {"symptom": "no ADC", "check": "gcloud auth application-default print-access-token", "gate": "authentication"},
    {"symptom": "object 403", "check": "maintainer read-only grant for exact identity", "gate": "authorization"},
    {"symptom": "requester pays", "check": "JOUVENCE_BILLING_PROJECT and billing enabled", "gate": "billing"},
    {"symptom": "service disabled/quota", "check": "storage.googleapis.com and serviceusage.services.use", "gate": "quota"},
])
display(troubleshooting)


### Interpretation

Do not solve a permission failure by switching to a broader identity or by downloading the bucket. Keep the test to one named object and use `scripts/verify_data_quickstart.py --mode live` as the bounded preflight.


In [ ]:
print({
    "preflight": "uv run python scripts/verify_data_quickstart.py --mode live",
    "canonical_root": str(PUBLIC_KG_ROOT),
    "forbidden_on_laptop": ["all-relation scan", "bulk download", "full PyG build", "full embedding scan"],
})


### Checkpoint

If the exact read remains blocked, return to fixture mode and report the failing gate. Do not infer that Jouvence data is absent or public just because one identity cannot read it.


## Navigate topology, evidence, metadata, features, and proof

`nodes/` stores typed entities; `edges/` stores deduplicated graph assertions; `evidence/` stores source records supporting those assertions; `metadata/` catalogs datasets and provenance objects; `features/` stores text, sequence, numeric, or embedding signals; optional `proof/` records reproducible derivations. Only nodes and edges define default adjacency.


In [ ]:
surface_contract = pd.DataFrame([
    ("nodes/", "stable typed entities", True),
    ("edges/", "deduplicated assertions", True),
    ("evidence/", "source-specific support", False),
    ("metadata/", "catalog and provenance objects", False),
    ("features/", "model/context signals", False),
    ("proof/", "deterministic derivation records", False),
], columns=["surface", "purpose", "changes_adjacency"])
display(surface_contract)


### Interpretation

A feature being canonical does not automatically authorize it for training. Its coverage, lineage, license, and leakage policy still govern use. Likewise, proof justifies a transformation but is not necessarily a model feature.


In [ ]:
catalog = parquet_catalog(KG_ROOT, billing_project=BILLING_PROJECT)
display(catalog[["layer", "path", "rows", "row_groups", "columns"]])
print("Catalog counts describe this selected root; fixture counts are illustrative only.")


### Checkpoint

Use the catalog to discover names and schemas before reading rows. Never guess a relation or embedding path when a manifest or catalog can identify it.


## Follow stable IDs, edge keys, and evidence multiplicity

Canonical human genes use Ensembl `ENSG...` IDs. Symbols, HGNC, NCBI Gene, UniProt, and source-native identifiers remain aliases or provenance. An edge key represents one assertion over relation and typed endpoints; multiple evidence records may support that key without duplicating topology.


In [ ]:
root = str(KG_ROOT).rstrip("/")
genes = read_bounded_parquet(f"{root}/nodes/gene.parquet", limit=4, billing_project=BILLING_PROJECT)
edges = read_bounded_parquet(f"{root}/edges/disease_associated_gene.parquet", limit=10, billing_project=BILLING_PROJECT)
evidence = read_bounded_parquet(f"{root}/evidence/disease_associated_gene.parquet", limit=20, billing_project=BILLING_PROJECT)
display(genes[[column for column in ["id", "gene_name", "hgnc_id", "ncbi_gene_id", "uniprot_id"] if column in genes]])


### Interpretation

Join assertions to evidence on `relation, x_id, x_type, y_id, y_type` or the validated edge key—not row position. Evidence count measures records represented, not automatically strength, independence, causality, or consensus.


In [ ]:
edge_keys = ["relation", "x_id", "x_type", "y_id", "y_type"]
joined = edges.merge(evidence, on=edge_keys, how="left", suffixes=("_assertion", "_evidence"), validate="one_to_many")
display(joined[[*edge_keys, "source_assertion", "source_evidence", "source_record_id"]].head(8))


### Checkpoint

Observed source assertions and controlled inferences must stay distinguishable in evidence or inferred-evidence surfaces. Do not relabel protein observations as RNA measurements or infer causal direction from association alone.


## Answer one bounded question and state its limits

We finish by asking which fixture diseases have represented TP53 association assertions and attached evidence. The reusable helper performs a local bounded join, labels entities, and preserves evidence summaries. This is the same reasoning path to adapt for another exact stable identifier.


In [ ]:
from manage_db.public_notebooks import diseases_with_gene_evidence
if MODE == "fixture":
    answer = diseases_with_gene_evidence(KG_ROOT, "ENSG00000141510", limit=20)
    display(answer)
else:
    answer = pd.DataFrame()
    print("For live work, first stage only reviewed bounded objects; the local DuckDB helper never scans the full remote KG.")


### Interpretation

What this means: the selected graph represents source-backed associations. What this does not mean: TP53 causes each disease, altering TP53 will treat it, evidence rows are independent, or omitted diseases are absent. Scores remain source-specific rather than calibrated clinical probabilities.


In [ ]:
summary = {
    "question": "Which represented diseases have TP53 association evidence?",
    "rows_returned": len(answer),
    "mode": MODE,
    "bounded": True,
    "next_notebook": "02_nodes_features_and_embeddings.ipynb",
}
print(summary)


### Checkpoint

Proceed to Notebook 02 to inspect versioned embeddings, or Notebook 03 to deepen the evidence join. Preserve the exact-ID, bounded-read, and interpretation-boundary habits throughout the course.
